# Análise de Consumo e Pedidos Scada-BPS

#### Calcular quantidade de dias de um intervalo:

In [34]:
from datetime import date, datetime

def calcular_dias_entre_datas(data_inicio_str, data_fim_str, formato="%d/%m/%Y"):
    """
    Calcula o número de dias entre duas datas.

    Args:
        data_inicio_str (str): A data de início no formato de string (ex: "01/01/2023").
        data_fim_str (str): A data final no formato de string (ex: "31/12/2023").
        formato (str): O formato da data utilizada nas strings. Padrão é "%d/%m/%Y".

    Returns:
        int: A diferença absoluta em dias entre as duas datas.
    """
    try:
        # Converte as strings de data para objetos datetime.date
        # O strptime() converte a string para datetime, e o .date() extrai apenas a data.
        data_inicio = datetime.strptime(data_inicio_str, formato).date()
        data_fim = datetime.strptime(data_fim_str, formato).date()

        # Calcula a diferença (o resultado é um objeto timedelta)
        diferenca = data_fim - data_inicio

        # Retorna o número de dias da diferença
        # Usamos abs() para garantir que o resultado seja sempre positivo,
        # independentemente da ordem das datas.
        return abs(diferenca.days)

    except ValueError as e:
        print(f"Erro ao converter a data. Verifique o formato '{formato}' e o valor das datas.")
        print(f"Detalhes do erro: {e}")
        return -1 # Retorna um valor de erro

# --- Exemplo de Uso ---
if __name__ == "__main__":
    # Defina as datas que você deseja calcular
    data_1 = "05/11/2025"
    data_2 = "04/12/2025"

    # Define o formato esperado (Dia/Mês/Ano)
    formato_data = "%d/%m/%Y"

    # Chama a função para calcular
    quantidade_dias = calcular_dias_entre_datas(data_1, data_2, formato_data) + 1

    if quantidade_dias != -1:
        print("-" * 40)
        print("CÁLCULO DE INTERVALO DE DATAS")
        print("-" * 40)
        print(f"Data de Início: {data_1}")
        print(f"Data Final: {data_2}")
        print(f"\nA quantidade de dias entre as duas datas é: {quantidade_dias} dias.")
        print("-" * 40)

    # Exemplo com datas em outra ordem
    data_3 = "05/11/2025"
    data_4 = "04/12/2025"
    dias_reverso = calcular_dias_entre_datas(data_3, data_4, formato_data)
    if dias_reverso != -1:
        print(f"Teste de ordem reversa ({data_3} a {data_4}): {dias_reverso} dias.")
        print("-" * 40)

    print(quantidade_dias)

----------------------------------------
CÁLCULO DE INTERVALO DE DATAS
----------------------------------------
Data de Início: 05/11/2025
Data Final: 04/12/2025

A quantidade de dias entre as duas datas é: 30 dias.
----------------------------------------
Teste de ordem reversa (05/11/2025 a 04/12/2025): 29 dias.
----------------------------------------
30


### Importação bases

In [35]:
import pandas as pd

consumo_df = pd.read_excel("base_consumo_cinema.xlsx")
contagem_df = pd.read_excel("base_contagem_semanal.xlsx")

display(consumo_df)
display(contagem_df)

,Cód. Item,Item,Unidade,Qtde.
0,BEBIDAS,NaN,NaN,NaN
1,57,AGUA COM GAS CIFAO,LT,74.95
2,1256,AGUA DE COCO,UN,11.00
3,186,AGUA GARRAFA C GAS,UN,252.00
4,185,AGUA GARRAFA S GAS,UN,224.00
...,...,...,...,...
149,UTENSILIOS,NaN,NaN,NaN
150,345,CANECA SCADA,UN,1.00
151,922,FILTRO CHEMEX CIRC.4 XICARA C/100 U.INT,UN,18.00
152,NaN,NaN,Total Grupo,19.00


,EST,L J,E+L\nunid,Fornecedor,Produto,Emb,Qtd/Emb,Total,Pedido,Vendido (Degust),Ped,Unnamed: 11,Relatório Degust,Unnamed: 13,Pedido Pendente,Unnamed: 15,Unnamed: 16,Variável produto,Unnamed: 18,Validação Cadastro item
0,NaN,NaN,-,BOTAFOGO E GELO,AGUA TONICA LATA,CX,12,0,NaN,1,0,NaN,13.000,NaN,NaN,1,NaN,NaN,NaN,188
1,NaN,NaN,-,BOTAFOGO E GELO,CERVEJA HEINEKEN LG NECK,CX,24,0,NaN,2,0,NaN,56.000,NaN,NaN,2,NaN,NaN,NaN,485
2,NaN,NaN,-,BOTAFOGO E GELO,COCA COLA LATA,CX,12,0,NaN,6,0,NaN,73.000,NaN,NaN,6,NaN,NaN,NaN,177
3,NaN,NaN,-,BOTAFOGO E GELO,COCA COLA ZERO LATA,CX,12,0,NaN,11,0,NaN,128.000,NaN,NaN,11,NaN,NaN,NaN,178
4,NaN,NaN,-,BOTAFOGO E GELO,GUARANA LATA,CX,12,0,NaN,2,0,NaN,20.000,NaN,NaN,2,NaN,NaN,NaN,182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,NaN,NaN,-,VINHOS,TAÇA ROSE PARCELAS (BLEND),UN,0.187,0,NaN,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,Não Localizado
149,NaN,NaN,-,VINHOS,TAÇA BRANCO PARCELAS (SAUVIGNON BLANC),UN,0.187,0,NaN,13,0,NaN,2.431,NaN,NaN,13,NaN,NaN,NaN,1137
150,NaN,NaN,-,NaN,NaN,NaN,NaN,0,NaN,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN
151,NaN,NaN,-,NaN,NaN,NaN,NaN,0,NaN,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN


### Tratamento dos dados

In [42]:
# Alterar nome de coluna Qtde:
consumo_df = consumo_df.rename(columns={"Qtde.": "Consumo período"})
contagem_df = contagem_df.rename(columns={"Validação Cadastro item": "Cód. Item"})


# Acrescentar coluna com cálculo de consumo médio diário:
consumo_df['Consumo Médio Dia'] = consumo_df['Consumo período'] / quantidade_dias
# Acrescentar coluna com a necessidade de estoque. inserir a quantidade de dias para suprimento e o percentual de segurança:
qtd_dias_pedido = 7
margem_seguranca_perc = 1.15
consumo_df['Necessidade Estoque'] = (consumo_df['Consumo Médio Dia'] * qtd_dias_pedido) * margem_seguranca_perc

# Agrupando bases a partir do código do item:
pedido_df = pd.merge(
    consumo_df[['Cód. Item', 'Item', 'Unidade', 'Consumo período', 'Consumo Médio Dia', 'Necessidade Estoque']], 
    contagem_df[['Cód. Item', 'Total']], 
    on='Cód. Item', how='outer'
    )

pedido_df['Pedido'] = pedido_df['Necessidade Estoque'] - pedido_df['Total']

# Arredondar as colunas numéricas:
colunas_arredondar = ["Consumo período", "Consumo Médio Dia", "Necessidade Estoque", "Total", "Pedido"]
pedido_df[colunas_arredondar] = pedido_df[colunas_arredondar].round(0)

#consumo_df.info()
#consumo_df.head(20)
display(pedido_df)

,Cód. Item,Item,Unidade,Consumo período,Consumo Médio Dia,Necessidade Estoque,Total,Pedido
0,2,CAPPUCCINO SCADA - 1KG,KG,8.0,0.0,2.0,0.0,2.0
1,4,LEITE INTEGRAL,LT,90.0,3.0,24.0,0.0,24.0
2,5,CHANTILY,LT,3.0,0.0,1.0,0.0,1.0
3,6,CAPPUCCINO DIET,KG,0.0,0.0,0.0,0.0,0.0
4,9,CALDA CHOCOLATE,KG,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...
238,NaN,NaN,Total Grupo,19.0,1.0,5.0,0.0,5.0
239,NaN,NaN,Total Grupo,19.0,1.0,5.0,0.0,5.0
240,NaN,NaN,Total,4934.0,164.0,1324.0,0.0,1324.0
241,NaN,NaN,Total,4934.0,164.0,1324.0,0.0,1324.0
